In [0]:
-- BUSINESS QUESTION:

-- Which products and departments are purchased most frequently?

-- This allows us to identify:
-- 1. The most frequently purchased products within each department, not just department totals
-- 2. How concentrated each department's purchase volume is — dominated by one hero product, or spread across many
-- 3. Which departments have no single dominant product, useful for merchandising/inventory decisions

-- Metric: purchase_count = number of times a product appears in Fact_Order_Products (one row per order-product line)
-- Metric: department_purchase_count = total purchase occurrences across all products in that department
-- Metric: product_share_pct = purchase_count as a percentage of department_purchase_count, showing how much of the department's purchase volume comes from this product

-- Dimensions:
--   department = product's department, from Dim_Product (merged in from departments)
--   product_name = individual product, from Dim_Product

-- This creates a table so the dashboard can read from it directly, instead of re-running the query each time, as per Sara's advice.

CREATE OR REPLACE TABLE AS
WITH product_purchases AS (
    SELECT
        dp.department,
        dp.product_name,
        COUNT(*) AS purchase_count   -- how many times this product was ordered, across all orders
    FROM `ftw-week-06`.`03-mart`.fact_order_products fop
    JOIN `ftw-week-06`.`03-mart`.dim_product dp
        ON fop.product_id = dp.product_id
    GROUP BY -- aggregate at product + department level first, before calculating department totals and ranking
        dp.department,
        dp.product_name
),

product_share AS (
    SELECT
        department,
        product_name,
        purchase_count,

        SUM(purchase_count) OVER (
            PARTITION BY department   -- total purchase occurrences for the whole department, repeated on every product row
        ) AS department_purchase_count,

        ROUND(
            purchase_count * 100.0 /
            SUM(purchase_count) OVER (
                PARTITION BY department
            ),
            2
        ) AS product_share_pct   -- this product's % share of its department's total purchase volume

    FROM product_purchases
),

ranked_products AS (
    SELECT
        department,
        product_name,
        purchase_count,
        department_purchase_count,
        product_share_pct,

        ROW_NUMBER() OVER (
            PARTITION BY department        -- restart the ranking for each department
            ORDER BY purchase_count DESC, product_name   -- rank most frequently purchased first, with product name as tiebreaker
        ) AS department_rank

    FROM product_share
)

SELECT
    department,
    product_name, CASE WHEN department_rank <= 5 THEN product_name ELSE 'Other' END AS product_category,
    purchase_count,
    department_purchase_count,
    product_share_pct,
    CASE WHEN department_rank <= 5 THEN department_rank ELSE 6 END AS display_rank
FROM ranked_products
WHERE department NOT IN ('Unknown', 'missing')--removed WHERE department_rank <= 3 will just filter in dashboard
ORDER BY -- sort by department, with the highest-frequency product first within each
    department,
    department_rank;

In [0]:
-- BUSINESS QUESTION:
-- How does customer purchasing behavior change by day of week and hour of day?
-- This allows us to identify:
-- 1. Which day has the most orders
-- 2. Which hours are busiest
-- 3. Which specific day + hour has the most orders
-- 4. Whether high order activity comes from many customers

-- Metric: order_count = number of unique orders
-- Dimensions:
--   order_dow = day of week
--   order_hour_of_day = hour of day
--   user_id/customer_count = number of unique customers

CREATE OR REPLACE TABLE `ftw-week-06`.`04-analytics`.order_by_day_hour AS

SELECT
    -- Convert the numeric DOW into a readable day name
    CASE
        WHEN order_dow = 0 THEN 'Sunday'
        WHEN order_dow = 1 THEN 'Monday'
        WHEN order_dow = 2 THEN 'Tuesday'
        WHEN order_dow = 3 THEN 'Wednesday'
        WHEN order_dow = 4 THEN 'Thursday'
        WHEN order_dow = 5 THEN 'Friday'
        WHEN order_dow = 6 THEN 'Saturday'
    END AS day_of_week,

    order_hour_of_day,    -- hour when the order was placed
    COUNT(DISTINCT order_id) AS order_count,    -- number of unique orders
    COUNT(DISTINCT user_id) AS customer_count   -- number of unique customers placing orders
FROM `ftw-week-06`.`03-mart`.dim_order

GROUP BY -- Analyze purchasing activity for every combination of day and hour
    order_dow,
    order_hour_of_day
ORDER BY -- Show the busiest purchasing periods first
    order_count DESC;

In [0]:
-- BUSINESS QUESTION:
-- Which products have the highest reorder behavior?
-- This allows us to identify:
-- 1. Which products are reordered most often?
-- 2. Which products have the highest reordered rate?
-- 3. Which departments have no single dominant product, useful for merchandising/inventory decisions
-- 4. What share of all purchased products are never reordered (one-time purchases) vs. reordered at least once?
-- Metric: reorder_rate = AVG(reordered) 
-- Metric: reorder_count = SUM(reordered)
-- Metric: Total_purchase_count = count(*)
-- Metric: one_time_purchase_flag
-- Dimensions:
--    product_id / product_name
--   product_name = individual product, from Dim_Product
-- This creates a table so the dashboard can read from it directly, instead of re-running the query each time.

CREATE OR REPLACE TABLE `ftw-week-06`.`04-analytics`.product_reorder_behavior AS
SELECT
    p.product_id,
    p.product_name,
    p.department,
    p.aisle,
    COUNT(*) AS total_orders,
    SUM(f.reordered) AS total_reorders,
    ROUND(AVG(f.reordered), 4) AS reorder_rate,
    CASE WHEN COUNT(*) >= 50 THEN TRUE ELSE FALSE END AS meets_min_sample_threshold,
    RANK() OVER (ORDER BY AVG(f.reordered) DESC) AS reorder_rate_rank
FROM `ftw-week-06`.`03-mart`.fact_order_products f
JOIN `ftw-week-06`.`03-mart`.dim_product p
    ON f.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.department,
    p.aisle
ORDER BY
    reorder_rate_rank;

--  Which products have the highest reorder count?
SELECT 
    p.product_name,
    SUM(f.reordered) AS total_reorders
FROM `ftw-week-06`.`03-mart`.fact_order_products f
JOIN `ftw-week-06`.`03-mart`.dim_product p
    ON f.product_id = p.product_id
GROUP BY 
    p.product_id, 
    p.product_name
ORDER BY 
    total_reorders DESC
LIMIT 10;

-- Highest reorder rate of products 
SELECT 
    p.product_id,
    p.product_name,
    COUNT(*) AS total_orders,
    SUM(f.reordered) AS total_reorders,
    ROUND(AVG(f.reordered), 4) AS reorder_rate
FROM `ftw-week-06`.`03-mart`.fact_order_products f
JOIN `ftw-week-06`.`03-mart`.dim_product p
    ON f.product_id = p.product_id
GROUP BY 
    p.product_id, 
    p.product_name
HAVING 
    COUNT(*) >= 50
ORDER BY 
    reorder_rate DESC
LIMIT 10;

-- Which departments/aisles have the highest overall reorder rate?
SELECT 
    p.department,
    p.aisle,
    COUNT(f.product_id) AS total_items_ordered,
    SUM(f.reordered) AS total_reorders,
    ROUND(AVG(f.reordered), 4) AS reorder_rate
FROM `ftw-week-06`.`03-mart`.fact_order_products f
JOIN `ftw-week-06`.`03-mart`.dim_product p
    ON f.product_id = p.product_id
GROUP BY 
    p.department, 
    p.aisle
HAVING 
    COUNT(f.product_id) >= 100
ORDER BY 
    reorder_rate DESC
LIMIT 10;

-- What share of all purchased products are never reordered vs. reordered at least once?
WITH product_status AS (
    SELECT 
        product_id,
        MAX(reordered) AS is_reordered
    FROM `ftw-week-06`.`03-mart`.fact_order_products
    GROUP BY product_id
)
SELECT
    CASE 
        WHEN is_reordered = 0 THEN 'Never Reordered (One-time only)'
        ELSE 'Reordered At Least Once' 
    END AS product_category,
    COUNT(*) AS product_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total_products
FROM product_status
GROUP BY 
    CASE 
        WHEN is_reordered = 0 THEN 'Never Reordered (One-time only)'
        ELSE 'Reordered At Least Once' 
    END;
